# AI Resume Screening System with LangChain & LangSmith Tracing

**Task 3 – GenAI Internship Assignment**

This notebook implements an end-to-end AI-powered Resume Screening System using:
- **LangChain** (LCEL pipelines, PromptTemplate)
- **HuggingFace Inference API** (Mistral-7B-Instruct)
- **LangSmith** (tracing & debugging)

**Pipeline:** Resume → Skill Extraction → Matching → Scoring → Explanation → Tracing

## 1. Install Dependencies

In [20]:
!pip install -q langchain langchain-huggingface langchain-core langsmith huggingface_hub

In [21]:
import os
from getpass import getpass

hf_token = getpass("Enter your HuggingFace API Token: ")
langsmith_key = getpass("Enter your LangSmith API Key: ")

os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
os.environ["LANGCHAIN_API_KEY"]         = langsmith_key
os.environ["LANGCHAIN_TRACING_V2"]      = "true"
os.environ["LANGCHAIN_PROJECT"]         = "ai-resume-screening"

print("Environment configured successfully.")

Enter your HuggingFace API Token: ··········
Enter your LangSmith API Key: ··········
Environment configured successfully.


In [22]:
import requests
y
headers = {"Authorization": f"Bearer {hf_token}"}

TEST_URL = "https://huggingface.co/api/whoami-v2"

def test_token_v2():
    response = requests.get(TEST_URL, headers=headers)
    if response.status_code == 200:
        user_info = response.json()
        print(f"✅ Token is valid! Logged in as: {user_info.get('name')}")
        print("Now try running the pipeline again. If it still fails with 400, the model ID might need a slight adjustment.")
    else:
        print(f"❌ Token Error: {response.status_code}")
        print(f"Details: {response.text}")

test_token_v2()

✅ Token is valid! Logged in as: Imamsab
Now try running the pipeline again. If it still fails with 400, the model ID might need a slight adjustment.


## 3. Sample Resumes & Job Description

In [23]:
# Three candidate resumes: strong, average, weak
strong_resume = """
Name: Imamsab
Experience: 4 years as Data Scientist at TechCorp
Skills: Python, Machine Learning, Deep Learning, TensorFlow, PyTorch, SQL, Pandas,
        Scikit-learn, NLP, Data Visualization, Statistics, A/B Testing, Docker
Education: M.Sc. Data Science, IIT Bombay
Projects:
- Built a churn prediction model (XGBoost) with 92% accuracy, saving $2M annually
- Deployed NLP sentiment pipeline on AWS for 1M+ daily users
- Led team of 3 on real-time recommendation engine using collaborative filtering
"""

average_resume = """
Name: SyadAli
Experience: 1.5 years as Junior Data Analyst at StartupXYZ
Skills: Python, SQL, Pandas, Excel, Tableau, basic Scikit-learn
Education: B.Tech Computer Science, VIT
Projects:
- Created sales dashboards in Tableau for weekly reporting
- Wrote SQL queries for data extraction and cleaning
- Built a simple linear regression model for sales forecasting
"""

weak_resume = """
Name: Moinuddin
Experience: 6 months internship in general IT support
Skills: Microsoft Office, basic Python (self-taught), HTML
Education: B.Sc. Mathematics (ongoing)
Projects:
- Managed company social media accounts
- Helped format internal HR documents
"""

# Job description for a Data Scientist role
job_description = """
Role: Data Scientist
Company: Analytics Corp

Requirements:
- 3+ years of data science experience
- Strong Python skills (Pandas, Scikit-learn, TensorFlow or PyTorch)
- Experience with SQL and data wrangling
- Knowledge of machine learning algorithms and model deployment
- Familiarity with NLP is a plus
- Experience with cloud platforms (AWS/GCP) preferred
- Strong communication and problem-solving skills
"""

# Dictionary of all candidates for batch processing
resumes = {
    "Strong Candidate (Aisha)": strong_resume,
    "Average Candidate (Rohan)": average_resume,
    "Weak Candidate (Priya)": weak_resume,
}

print("Resumes and job description loaded.")
print(f"Total candidates to screen: {len(resumes)}")

Resumes and job description loaded.
Total candidates to screen: 3


## 4. Load HuggingFace LLM via ChatHuggingFace

We use `ChatHuggingFace` wrapping `HuggingFaceEndpoint`. This:
- Properly handles the **conversational** task routing that HuggingFace Inference API uses for instruct models
- Works with LangChain's LCEL pipe `|` syntax
- Is fully traced by LangSmith

In [24]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.output_parsers import StrOutputParser
import os

# Adding task parameter to ensure the Inference API routes correctly
endpoint = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    max_new_tokens=512,
    temperature=0.1,
    task="text-generation",
    huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN")
)

# ChatHuggingFace handles the conversational formatting
llm = ChatHuggingFace(llm=endpoint)

# StrOutputParser extracts the string content
parser = StrOutputParser()

print("LLM updated with explicit task routing. Please run the pipeline cell (RzRnGfe5GP23) now.")

LLM updated with explicit task routing. Please run the pipeline cell (RzRnGfe5GP23) now.


## 5. Prompts

Three modular prompts for the pipeline:
1. **Extraction** – parse structured info out of raw resume text
2. **Matching** – compare extracted profile against job requirements
3. **Scoring** – assign 0-100 fit score with explanation

In [25]:

from langchain_core.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""\
[INST]
You are a resume parser. Extract ONLY information explicitly present in the resume.
Do NOT assume or infer anything not directly stated.

Resume:
{resume}

Return exactly in this format:
- Skills: (comma-separated list)
- Years of Experience: (number)
- Tools & Technologies: (comma-separated list)
- Education: (degree and institution)
[/INST]"""
)

# ── Step 2: Matching Logic Prompt ────────────────────────────────────────────
# Compares candidate profile vs job requirements — structured diff output.
matching_prompt = PromptTemplate(
    input_variables=["extracted_info", "job_description"],
    template="""\
[INST]
You are a recruiter assistant. Compare the candidate profile against the job requirements.
Only use the information provided. Do NOT assume additional skills.

Candidate Profile:
{extracted_info}

Job Description:
{job_description}

Return exactly in this format:
- Matched Requirements: (list what the candidate meets)
- Missing Requirements: (list what the candidate lacks)
- Bonus Qualifications Met: (any extra positives beyond the requirements)
[/INST]"""
)


scoring_prompt = PromptTemplate(
    input_variables=["match_analysis", "job_description"],
    template="""\
[INST]
You are a hiring evaluator. Based on the match analysis below, assign a fit score from 0 to 100.
Only use what is in the analysis — no assumptions.

Match Analysis:
{match_analysis}

Job Description:
{job_description}

Scoring guide:
- 80-100: Meets most or all requirements
- 50-79:  Meets some requirements
- 0-49:   Meets few requirements

Return exactly:
Fit Score: <number>/100
Explanation: <2-3 sentences explaining the score based strictly on the match analysis>
[/INST]"""
)

print("All prompts defined successfully.")

All prompts defined successfully.


## 6. Build LCEL Chains

Using **LangChain Expression Language (LCEL)** pipe `|` syntax — each chain is:
`PromptTemplate → ChatHuggingFace LLM → StrOutputParser`

In [26]:
extraction_chain = extraction_prompt | llm | parser   # Step 1: Extract
matching_chain   = matching_prompt   | llm | parser   # Step 2: Match
scoring_chain    = scoring_prompt    | llm | parser   # Step 3 & 4: Score + Explain

print("LCEL chains created:")
print("  extraction_chain → extraction_prompt | llm | parser")
print("  matching_chain   → matching_prompt   | llm | parser")
print("  scoring_chain    → scoring_prompt    | llm | parser")

LCEL chains created:
  extraction_chain → extraction_prompt | llm | parser
  matching_chain   → matching_prompt   | llm | parser
  scoring_chain    → scoring_prompt    | llm | parser


## 7. Full Screening Pipeline Function

In [27]:
def run_screening_pipeline(candidate_name, resume, job_description):
    """
    Run the full 4-step AI resume screening pipeline for one candidate.
    Each step is automatically traced in LangSmith.

    Args:
        candidate_name  (str): Display name of the candidate
        resume          (str): Raw resume text
        job_description (str): Job description text

    Returns:
        dict: Extracted info, match analysis, and score/explanation
    """
    print(f"\n{'='*60}")
    print(f"Screening: {candidate_name}")
    print('='*60)

    # ── Step 1: Skill Extraction ──────────────────────────────────
    print("\n[Step 1] Extracting skills and experience...")
    extracted = extraction_chain.invoke({"resume": resume})
    print(extracted)

    # ── Step 2: Matching Logic ────────────────────────────────────
    print("\n[Step 2] Matching against job requirements...")
    match_result = matching_chain.invoke({
        "extracted_info": extracted,
        "job_description": job_description
    })
    print(match_result)

    # ── Step 3 & 4: Scoring + Explanation ────────────────────────
    print("\n[Step 3 & 4] Scoring and explaining...")
    score_result = scoring_chain.invoke({
        "match_analysis": match_result,
        "job_description": job_description
    })
    print(score_result)

    return {
        "candidate": candidate_name,
        "extracted_info": extracted,
        "match_analysis": match_result,
        "score_and_explanation": score_result
    }

print("Pipeline function defined and ready.")

Pipeline function defined and ready.


## 8. Run All Three Candidates

Each `.invoke()` call is **automatically traced in LangSmith** — check your project at https://smith.langchain.com

In [35]:
# Screen all three candidates — each run appears in LangSmith dashboard
all_results = []

for candidate_name, resume in resumes.items():
    result = run_screening_pipeline(candidate_name, resume, job_description)
    all_results.append(result)

print("\nAll candidates screened. Check LangSmith dashboard for traces.")

## 9. Summary Table

In [30]:
print("\n" + "="*60)
print("FINAL SCREENING SUMMARY")
print("="*60)
print(f"{'Candidate':<35} {'Fit Score'}")
print("-"*55)

for result in all_results:
    score_text = result["score_and_explanation"]
    # Extract the Fit Score line from the scoring output
    score_line = next(
        (line for line in score_text.split("\n") if "Fit Score" in line),
        "Score: N/A"
    )
    print(f"{result['candidate']:<35} {score_line.strip()}")

print("="*60)


FINAL SCREENING SUMMARY
Candidate                           Fit Score
-------------------------------------------------------


## 10. Bonus — Structured JSON Output

Returns a structured JSON evaluation for each candidate — useful for downstream processing.

In [34]:
import json

# JSON output prompt — forces the LLM to respond in a parseable JSON structure
json_prompt = PromptTemplate(
    input_variables=["resume", "job_description"],
    template="""\
[INST]
Evaluate this resume against the job description.
Return ONLY a valid JSON object — no markdown, no text outside the JSON.
Only use information explicitly present in the resume.

Resume: {resume}
Job Description: {job_description}

JSON format:
{{
  "candidate_name": "<name from resume>",
  "fit_score": <0-100>,
  "matched_skills": ["skill1", "skill2"],
  "missing_skills": ["skill1", "skill2"],
  "years_of_experience": "<from resume>",
  "explanation": "<2 sentences>",
  "recommendation": "<Shortlist / Consider / Reject>"
}}
[/INST]"""
)

# Build the JSON chain using LCEL
json_chain = json_prompt | llm | parser

print("Running JSON output pipeline on all candidates...\n")

for candidate_name, resume in resumes.items():
    print(f"--- {candidate_name} ---")
    raw = json_chain.invoke({"resume": resume, "job_description": job_description})
    try:
        # Strip any preamble text before the first '{' in case the model adds it
        json_start = raw.find("{")
        clean = raw[json_start:] if json_start != -1 else raw
        parsed = json.loads(clean)
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print("[JSON parse error — raw output below]")
        print(raw)
    print()

## 11. Debug — Intentional Incorrect Output (LangSmith Debug Case)

This cell runs a **deliberately vague prompt** on purpose — to demonstrate in LangSmith why structured prompts matter. The bad output will appear alongside the structured ones in the LangSmith dashboard.

In [33]:
# Intentionally bad prompt — no format, no context, no constraints
# This is a LangSmith debugging demonstration: shows why prompt engineering matters

bad_prompt = PromptTemplate(
    input_variables=["resume"],
    template="[INST] Evaluate this person and give them a score. Resume: {resume} [/INST]"
    # Problems with this prompt:
    # 1. No output format specified → hallucinated/inconsistent output
    # 2. No job description context → score is meaningless
    # 3. No scoring scale defined → model invents its own scale
)

bad_chain = bad_prompt | llm | parser

print("[DEBUG] Running intentionally bad prompt on weak candidate...")
bad_output = bad_chain.invoke({"resume": weak_resume})
print("\nBad output (for LangSmith debugging demo):")
print(bad_output)
print("\n=> This run appears in LangSmith showing why structured prompts are essential.")